# Глава 15. Модель в работе: MLflow, FastAPI, Docker — практика в Jupyter

Те же задачи, что на сайте курса, но в своём Jupyter: так вы привыкаете к блокноту, горячим клавишам и своему окружению. Проверка — та же, что в браузере.

**Как запустить** (один раз поставьте [uv](https://docs.astral.sh/uv/), см. урок «Своё рабочее место — uv и Jupyter» главы 2). В терминале, в папке курса:

```sh
uv run --with-requirements site/notebooks/requirements.txt jupyter lab site/notebooks
```

Без uv: `python3 -m venv .venv`, затем `.venv/bin/pip install -r site/notebooks/requirements.txt` (в Windows — `.venv\Scripts\pip`) и `.venv/bin/jupyter lab site/notebooks`. Откройте этот файл в папке `site/notebooks` — рядом с ним лежит расширение `dsc.py`, без него проверка не заработает.

**Как решать**

1. Запустите ячейку ниже с `%load_ext dsc` — она подключает проверку. После перезапуска ядра запустите её снова.
2. В ячейке задачи первая строка — `%%task глава/урок/id`. Её не трогайте, пишите код ниже и запускайте **Shift+Enter**. Под выводом появится «✓ Верно» или подсказка, что не так.
3. Переменные живут между ячейками, как в любом блокноте. Исключение — переменные, в которые задача просит положить ответ: перед запуском ячейки они стираются, чтобы проверка видела результат именно этого запуска. Подготовка задачи (если она есть) выполняется перед вашим кодом при каждом запуске.
4. Застряли — `dsc.hint('id')` покажет следующую подсказку, `dsc.solution('id', sure=True)` — эталон с разбором. `dsc.progress()` — что решено в этом сеансе.
5. Файлы данных берутся по имени, как на сайте: `pd.read_csv('orders.csv')`. Код работает в папке `site/notebooks/_work` — там копия `site/data`, и туда же попадает всё, что сохраняет ваш код.

Прогресс сайта и блокнота не связаны: отметки о решённых задачах на сайте ставятся только там. Чтобы начать с чистого листа, скачайте блокнот заново или возьмите его из архива курса.

In [ ]:
%load_ext dsc

## Урок 1. Модель как артефакт: файл, паспорт, контрольный набор

Модель, которая живёт только в ноутбуке аналитика, бизнесу не помогает. Чтобы ей пользовались другие — CRM, разработчики, коллега через полгода, — её выпускают как артефакт: файл модели, паспорт с описанием, контрольный набор для проверки и номер версии. Разбираем каждую часть по шагам на модели оттока гостей «Зерна».

Теория — на сайте курса: `ch15/01-artifact.html`. Задачи «собери код из строк» (1) остались на сайте.

### Отпечаток строки · разминка

Посчитайте MD5-отпечатки строк `"Зерно"` и `"зерно"` (с маленькой буквы): `h1` и `h2` — строки из 32 символов (`hexdigest`). В `same` — равны ли они.

*Подсказки (3): `dsc.hint('md5-text')`*

In [ ]:
%%task ch15/01-artifact/md5-text
import hashlib

h1 = ...
h2 = ...
same = ...
print(h1, h2, same)

### Сколько весит модель · разминка

Модели `small` (логистическая регрессия) и `big` (лес из 200 деревьев с `min_samples_leaf=5`) обучены в заготовке. Сохраните их в файлы `small.joblib` и `big.joblib` и запишите в `kb_small` и `kb_big` их размеры в килобайтах (байты / 1024, округлить до 1 знака). В `ratio` — во сколько раз лес тяжелее (целое, `round`).

*Подсказки (2): `dsc.hint('size')`*

In [ ]:
%%task ch15/01-artifact/size
import os
import joblib

### Отпечаток файла · база

Напишите функцию `file_md5(path)`, которая возвращает MD5-отпечаток файла (строка `hexdigest`). Сохраните `data_md5` — отпечаток `guests_churn.csv`. Затем создайте копию файла `guests_copy.csv`, в которой изменено одно число: сохраните `pd.read_csv("guests_churn.csv")` с `tenure_months` первой строки, увеличенным на 1, через `to_csv(..., index=False)`. В `copy_md5` — её отпечаток, в `changed` — отличаются ли отпечатки.

*Подсказки (3): `dsc.hint('data-md5')`*

In [ ]:
%%task ch15/01-artifact/data-md5
import hashlib
import pandas as pd

def file_md5(path):
    ...

### Паспорт модели · база

Модель `model` обучена в заготовке (как в уроке), `X_test`, `y_test` — тест. Соберите словарь `card` с ключами:

- `"name"`: `"churn"`, `"version"`: `"1.0"`;
- `"features"`: список имён признаков из `model.feature_names_in_` (обычный список строк);
- `"threshold"`: `1 / 3`;
- `"auc_test"`: AUC на тесте (3 знака, обычный `float`); для простоты — прямо в `card`, без вложенного словаря `"metrics"`, как в теории;
- `"n_train"`: число обучающих строк (обычный `int`);
- `"data_md5"`: отпечаток `guests_churn.csv`;
- `"sklearn"`: версия sklearn.

Сохраните его в `card.json` (`ensure_ascii=False, indent=2`), загрузите обратно в `card2` и проверьте `card2 == card` — результат в `same`.

*Подсказки (3): `dsc.hint('card')`*

In [ ]:
%%task ch15/01-artifact/card
import json
import hashlib
import sklearn
from sklearn.metrics import roc_auc_score

card = {
    "name": "churn",
    "version": "1.0",
    ...
}

### Контрольный набор · база

Модель `model` и тест `X_test` — в заготовке. Сохраните контрольный набор `golden.csv`: первые 20 строк `X_test` и столбец `expected` с вероятностью ухода. Сохраните модель в `model.joblib`, загрузите в `loaded` и посчитайте `max_diff` — наибольшую абсолютную разницу между ответами `loaded` на строки из `golden.csv` и столбцом `expected`. В `passed` — `max_diff < 1e-9`.

*Подсказки (3): `dsc.hint('ch15/01-artifact/golden')`*

In [ ]:
%%task ch15/01-artifact/golden
import numpy as np
import pandas as pd
import joblib

### Проверка входа · база

Напишите `check_input(df)` по образцу из урока: вернуть список строк-проблем (пустой — всё хорошо). Проверьте: отсутствующие столбцы (сообщение начинается с `"нет столбцов"`), лишние (`"лишние столбцы"`), нечисловые числовые признаки (`"<столбец>: ожидаются числа"`), незнакомые значения в `city` и `fav_category` (`"<столбец>: незнакомые значения"`). Известные значения возьмите из обученного кодировщика `model` (заготовка). Примените к трём выгрузкам из заготовки: `p_ok`, `p_bad1`, `p_bad2` — списки проблем.

*Подсказки (3): `dsc.hint('ch15/01-artifact/contract')`*

In [ ]:
%%task ch15/01-artifact/contract
import pandas as pd

FEATURES = [str(f) for f in model.feature_names_in_]
enc = model.named_steps["prep"].named_transformers_["cat"]
KNOWN = {name: [str(v) for v in cats] for name, cats in zip(CAT, enc.categories_)}

def check_input(df):
    problems = []
    ...
    return problems

### Решение по паспорту · посложнее

Напишите `decide(model, card, df)`: на вход — модель, паспорт (словарь) и таблица гостей **с** `guest_id`. Вернуть DataFrame с колонками `guest_id`, `proba` (вероятность ухода, 3 знака) и `in_campaign` (`True`, если `proba` без округления ≥ `card["threshold"]`), отсортированный по `proba` по убыванию. Признаки берите строго из `card["features"]` в этом порядке. Примените к первым 300 гостям файла — результат в `campaign`, и сохраните `n_in` — сколько гостей попало в акцию.

*Подсказки (3): `dsc.hint('decide')`*

In [ ]:
%%task ch15/01-artifact/decide
import pandas as pd

def decide(model, card, df):
    ...

campaign = decide(model, card, new_guests)
n_in = ...
print(campaign.head(), n_in)

### Что изменит новая версия · посложнее

В заготовке обучены `v10` (логистическая регрессия) и `v11` (лес). Для тестовых гостей посчитайте решения обеих версий при пороге 1/3. Сохраните `auc10`, `auc11` (3 знака), `n10`, `n11` — сколько гостей в акции у каждой версии, `flip` — долю гостей, у которых решение изменилось (3 знака), `newly_in` — сколько гостей попадёт в акцию только в 1.1, и `thr11` — порог для версии 1.1, при котором в акцию попадёт столько же гостей, сколько в 1.0 (возьмите `n10`-ю по величине вероятность версии 1.1, 3 знака).

*Подсказки (3): `dsc.hint('ch15/01-artifact/versions')`*

In [ ]:
%%task ch15/01-artifact/versions
import numpy as np
from sklearn.metrics import roc_auc_score

p10 = v10.predict_proba(X_test)[:, 1]
p11 = v11.predict_proba(X_test)[:, 1]

### Проверка перед выпуском · со звёздочкой

Соберите функцию `release_check(model, card, golden_path, X_test, y_test)`, которая возвращает словарь из трёх проверок (`True`/`False`):

- `"features"` — `list(model.feature_names_in_)` совпадает с `card["features"]`;
- `"golden"` — расхождение с контрольным набором из `golden_path` меньше `1e-9`;
- `"auc"` — AUC модели на тесте не ниже `card["auc_test"] − 0.01`.

В заготовке есть модель `good`, её паспорт `card` и контрольный набор `golden.csv`, а также модель `retrained` — та же схема, но обученная на другой половине данных (кто-то «случайно» переобучил и положил в ту же папку). Сохраните `res_good = release_check(good, ...)` и `res_bad = release_check(retrained, ...)`.

*Подсказки (3): `dsc.hint('release')` · задача по желанию*

In [ ]:
%%task ch15/01-artifact/release
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

def release_check(model, card, golden_path, X_test, y_test):
    ...

## Урок 2. Модель как сервис: запрос, проверка, ответ

Приложение «Зерна» хочет спрашивать модель про одного гостя прямо в момент, когда он открывает экран бонусов. Значит, модель должна стать сервисом: принимать запрос в JSON, проверять его и отвечать понятным кодом, а не падать от кривых данных. Строим такой обработчик по шагам как обычную функцию Python. Пакет запросов, время ответа, FastAPI и тесты — в следующем уроке.

Теория — на сайте курса: `ch15/02-service.html`. Задачи «собери код из строк» (1) остались на сайте.

### Туда и обратно · разминка

Строка `request_text` — запрос от приложения. Превратите её в словарь `payload`, измените в нём `visits_90d` на 5 и превратите обратно в JSON-текст `out_text` с русскими буквами без экранирования. В `has_null` — есть ли в `out_text` подстрока `null`.

*Подсказки (2): `dsc.hint('json-rt')`*

In [ ]:
%%task ch15/02-service/json-rt
import json

### Ловушка с True · разминка

Напишите функции `is_int(x)` (целое, но не логическое) и `is_number(x)` (целое или дробное, но не логическое). Посчитайте `results` — список значений `is_int` для `[5, True, 5.0, "5", -2, False]`.

*Подсказки (2): `dsc.hint('is-int')`*

In [ ]:
%%task ch15/02-service/is-int
def is_int(x):
    ...

def is_number(x):
    ...

results = [is_int(v) for v in [5, True, 5.0, "5", -2, False]]
print(results)

### Проверка запроса · база

Напишите `validate(p)` по контракту из урока (таблица в разделе «Проверка входа»): вернуть список строк вида `"поле: ..."`, по одной на каждое нарушение, пустой — если всё в порядке. Обязательно: отсутствующие поля, неизвестные город и категория, целые поля (тип и диапазон), `avg_check` (`None` или число 50–3000), `delivery_share` (число 0–1). Константы `CITIES`, `CATEGORIES`, `INT_RANGES` и функции `is_int`, `is_number` — в заготовке. Проверка прогонит вашу функцию на десятке запросов и посмотрит, **какие поля** названы в ошибках.

*Подсказки (3): `dsc.hint('ch15/02-service/validate')`*

In [ ]:
%%task ch15/02-service/validate
def validate(p):
    errors = []
    ...
    return errors

print(validate(GOOD))
print(validate(dict(GOOD, visits_90d=-1)))

### Обработчик с кодами · база

Модель `model`, паспорт `CARD` и готовая `validate` — в заготовке. Напишите `predict_one(payload)`:

- ошибки проверки → `{"status": 422, "errors": [...]}`;
- иначе посчитать вероятность; если при этом возникло исключение → `{"status": 500, "errors": ["<ИмяОшибки>: <текст>"]}`;
- успех → `{"status": 200, "proba": <3 знака>, "in_campaign": <bool>, "model_version": CARD["version"]}`.

Сохраните `r_good`, `r_bad` — ответы для `GOOD` и для `GOOD` без поля `city`.

*Подсказки (3): `dsc.hint('handler')`*

In [ ]:
%%task ch15/02-service/handler
import pandas as pd

def predict_one(payload):
    ...

r_good = predict_one(GOOD)
r_bad = predict_one({k: v for k, v in GOOD.items() if k != "city"})
print(r_good, r_bad, sep="\n")

### Текст на входе, текст на выходе · база

Сервис получает не словарь, а **текст** запроса. Напишите `handle(text)`: разобрать JSON; если текст — не JSON или не объект (`{...}`) → вернуть текст `{"status": 400, "errors": ["тело запроса — не JSON-объект"]}`; иначе вызвать готовую `predict_one` и вернуть её ответ как JSON-текст (`ensure_ascii=False`). Сохраните ответы на три запроса из заготовки: `a1`, `a2`, `a3` (строки).

*Подсказки (2): `dsc.hint('http')`*

In [ ]:
%%task ch15/02-service/http
import json

def handle(text):
    ...

a1, a2, a3 = handle(q1), handle(q2), handle(q3)
print(a1, a2, a3, sep="\n")

### Ответ с причинами · со звёздочкой

Оператор поддержки видит в CRM «вероятность ухода 0.87» и спрашивает: почему? Для логистической регрессии ответ можно посчитать: вклад признака = значение признака после подготовки × его вес (глава 8). Напишите `explain_one(payload, k=2)`, которая возвращает список из `k` названий признаков с наибольшим **положительным** вкладом (те, что сильнее всего тянут к уходу), по убыванию вклада. Названия — как в `get_feature_names_out()` подготовки, но без префикса до `__` (`"num__days_since_last"` → `"days_since_last"`, `"cat__city_Тюмень"` → `"city_Тюмень"`). Сохраните `top_guest` — ответ для гостя `GUEST` из заготовки.

*Подсказки (3): `dsc.hint('reasons')` · задача по желанию*

In [ ]:
%%task ch15/02-service/reasons
import numpy as np
import pandas as pd

def explain_one(payload, k=2):
    prep = model.named_steps["prep"]
    lr = model.named_steps["model"]
    ...

top_guest = explain_one(GUEST)
print(top_guest)

## Урок 3. Сервис в работе: пакет, время ответа, FastAPI, тесты

Обработчик с проверкой входа и кодами ответа готов — теперь доводим его до работы. Пакет запросов, в котором одна плохая строка не роняет остальные; время ответа в процентилях p50 и p95 и цена одиночного вызова против пакета; та же логика как веб-сервис на FastAPI и pydantic; тесты, которые ловят дыры в проверке до выкладки.

Теория — на сайте курса: `ch15/02b-service-run.html`.

### Пакет без падений · база

Готовая `predict_one` — в заготовке (полная версия: проверка + модель). В `batch_requests` — 50 запросов, среди которых есть плохие. Напишите `predict_batch(payloads)`, возвращающий DataFrame ответов в том же порядке, и примените: `res`. Сохраните `status_counts` — словарь `{код: число}` и `bad_fields` — Series: сколько раз каждое поле встречается в ошибках 422 (поле — часть строки ошибки до двоеточия), по убыванию.

*Подсказки (3): `dsc.hint('batch')`*

In [ ]:
%%task ch15/02b-service-run/batch
import pandas as pd

def predict_batch(payloads):
    ...

res = predict_batch(batch_requests)

### Хвост в цифрах · разминка

В заготовке `times` — время ответа сервиса на 20 запросов в миллисекундах. Требование к сервису: «p95 не больше 100 мс». Посчитайте `p50`, `p95` (через `np.percentile`), `mean_ms` — среднее, `slow` — сколько запросов дольше 50 мс, и `meets` — выполнено ли требование (обычный `bool`).

*Подсказки (2): `dsc.hint('ch15/02b-service-run/tail')`*

In [ ]:
%%task ch15/02b-service-run/tail
import numpy as np

p50 = ...

### Время ответа · посложнее

Модель `model` и таблица `X` — в заготовке. Замерьте время `model.predict_proba` на одной строке для первых 200 гостей (по одному вызову на гостя) и время одного вызова на всех 200 сразу. Сохраните `times` — список из 200 времён в миллисекундах, `p50`, `p95` (миллисекунды, `np.percentile`), `mean_ms` — среднее, `batch_per_row` — время пакета, делённое на 200, и `speedup` — во сколько раз пакет быстрее на строку (`p50 / batch_per_row`).

*Подсказки (3): `dsc.hint('latency')`*

In [ ]:
%%task ch15/02b-service-run/latency
import time
import numpy as np

times = []

### Найти дыру в проверке · посложнее

В заготовке — функция `validate_v0` (её написал коллега) и список тестовых случаев `cases`: `(название, запрос, ожидаемый статус)`. Статус по проверке: 422, если `validate_v0` вернула ошибки, иначе 200. Прогоните все случаи и сохраните `failed` — список названий случаев, где статус не совпал с ожидаемым. Затем напишите исправленную `validate_v1` (на основе `validate_v0`), которая проходит все случаи, — `failed_v1` должен быть пустым.

*Подсказки (3): `dsc.hint('tests')`*

In [ ]:
%%task ch15/02b-service-run/tests
def status_of(validate, payload):
    return 422 if validate(payload) else 200

failed = ...

def validate_v1(p):
    ...

failed_v1 = [name for name, p, exp in cases if status_of(validate_v1, p) != exp]
print(failed, failed_v1)

### Тесты, которые ловят ошибки · база

В заготовке — исправленная проверка `predict_ok` и две версии с ошибками, которые однажды попадали в работу: в `predict_bug_bool` поле `app` принимает `True`, в `predict_bug_share` доля доставки проверяется до 100, а не до 1. Все три возвращают словарь `{"status": ...}`. Функция `check_status` — из раздела «Тесты сервиса», запрос `GOOD` — нормальный гость.

Составьте список `cases` из кортежей `(название, запрос, ожидаемый статус)` — не меньше трёх случаев. Требования: на `predict_ok` все случаи проходят (`check_status(predict_ok, cases) == []`), а каждая версия с ошибкой проваливает хотя бы один случай.

*Подсказки (2): `dsc.hint('tests-write')`*

In [ ]:
%%task ch15/02b-service-run/tests-write
cases = [
    ("норма", GOOD, 200),
    # добавьте случаи, на которых ошибаются версии с багами
]

print(check_status(predict_ok, cases))
print(check_status(predict_bug_bool, cases))
print(check_status(predict_bug_share, cases))

## Урок 4. Учёт экспериментов и воспроизводимость

За месяц работы над моделью набираются десятки запусков: разные модели, параметры, признаки, версии данных. Без журнала через неделю не вспомнить, какой запуск был лучшим и можно ли его повторить. Ведём журнал экспериментов руками, учимся отличать настоящее улучшение от случайного шума и знакомимся с MLflow — инструментом, который делает то же самое автоматически.

Теория — на сайте курса: `ch15/03-experiments.html`. Задачи «собери код из строк» (1) остались на сайте.

### Первая запись журнала · разминка

Создайте пустой список `runs`, добавьте в него два словаря-запуска с ключами `run_id`, `model`, `cv_auc` — `("run-001", "LogReg", 0.875)` и `("run-002", "Forest", 0.901)` — и превратите в таблицу `journal`. В `best_id` — `run_id` строки с наибольшим `cv_auc`.

*Подсказки (2): `dsc.hint('run-dict')`*

In [ ]:
%%task ch15/03-experiments/run-dict
import pandas as pd

runs = []

### Одно зерно — один результат · разминка

Обучите три леса `RandomForestClassifier(30, random_state=...)` на `X`, `y` из заготовки: два с `random_state=7` и один с `random_state=8`. Сохраните `same_7` — совпадают ли (`np.array_equal`) их вероятности на первых 20 гостях у двух лесов с зерном 7, и `same_7_8` — у лесов с зёрнами 7 и 8. В `max_gap` — наибольшую разницу вероятностей между лесами 7 и 8 на этих 20 гостях (3 знака).

*Подсказки (2): `dsc.hint('seed')`*

In [ ]:
%%task ch15/03-experiments/seed
import numpy as np
from sklearn.ensemble import RandomForestClassifier

### Журнал на диске · база

Напишите `append_run(run, path)`: дописать словарь `run` одной строкой в CSV (`mode="a"`, заголовок — только если файла ещё нет, без индекса). Удалите старый `journal.csv`, если он есть, и запишите в него три запуска из заготовки (`r1`, `r2`, `r3`). Прочитайте файл в `back`. В `n_header_lines` — сколько строк файла начинается с `run_id` (должна быть одна).

*Подсказки (3): `dsc.hint('log-csv')`*

In [ ]:
%%task ch15/03-experiments/log-csv
import os
import pandas as pd

def append_run(run, path):
    ...

if os.path.exists("journal.csv"):
    os.remove("journal.csv")

### Серия запусков · база

Функции `make(estimator)` (конвейер) и фолды `CV` — в заготовке. Проведите серию запусков и соберите журнал `journal` (DataFrame) со столбцами `model`, `param`, `cv_auc_mean`, `cv_auc_std` (4 знака) — по строке на запуск:

- логистическая регрессия с `C` из `[0.01, 0.1, 1, 10]` (`max_iter=2000`), `param` — значение `C`;
- лес `RandomForestClassifier(100, max_depth=d, min_samples_leaf=5, random_state=0)` с `d` из `[3, 6]`, `param` — глубина.

Оценка — `cross_val_score(make(est), X_train, y_train, cv=CV, scoring="roc_auc")`. Сохраните `best` — строку журнала (Series) с наибольшим средним.

*Подсказки (3): `dsc.hint('ch15/03-experiments/grid')`*

In [ ]:
%%task ch15/03-experiments/grid
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

runs = []

def log(model_name, param, est):
    ...

### Парное сравнение · база

Сравните лес `RandomForestClassifier(100, max_depth=6, min_samples_leaf=5, random_state=0)` и логистическую регрессию (`max_iter=2000`) по одним и тем же фолдам `CV`. Сохраните `diff` — массив разностей AUC «лес − регрессия» по 5 фолдам, `mean_diff` (4 знака), `wins` — в скольких фолдах лес лучше, и `sd_lr` — разброс AUC регрессии по фолдам (`std`, 4 знака). В `paired_clear` — `True`, если лес лучше во всех пяти фолдах.

*Подсказки (2): `dsc.hint('ch15/03-experiments/paired')`*

In [ ]:
%%task ch15/03-experiments/paired
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

### Шум от зерна · посложнее

Оцените, сколько «шумит» лес из-за случайности. Для `random_state` от 0 до 5 посчитайте среднее AUC кросс-валидации леса `RandomForestClassifier(60, min_samples_leaf=5, random_state=s)` (фолды `CV`). Сохраните `by_seed` — список из 6 средних, `seed_sd` — их стандартное отклонение (4 знака), `seed_range` — размах (макс − мин, 4 знака). Затем сравните с разницей двух глубин из журнала: `depth_gap` — разность средних AUC лесов с `max_depth=6` и `max_depth=3` (оба `random_state=0`, 60 деревьев, `min_samples_leaf=5`), 4 знака. В `gap_within_range` — `True`, если `abs(depth_gap)` меньше `seed_range`, то есть разница глубин укладывается в размах от смены зерна.

Здесь обучается 8 лесов по 5 фолдов — 40 моделей. В браузере расчёт может занять около минуты: это не зависание, дождитесь вывода.

*Подсказки (2): `dsc.hint('seed-noise')`*

In [ ]:
%%task ch15/03-experiments/seed-noise
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

def cv_auc(est):
    return cross_val_score(make(est), X_train, y_train, cv=CV, scoring="roc_auc").mean()

### Удачливая пустышка · посложнее

Повторите опыт с 40 случайными «моделями» на маленькой валидации и проверьте, как меняется иллюзия с размером валидации. Для `n` из `[50, 200, 1000]`: валидация — первые `n` значений `churned`; «модель» с номером `s` выдаёт `np.random.default_rng(s).random(n)`, `s` от 0 до 39. Сохраните `best_by_n` — словарь `{n: наибольший AUC среди 40 пустышек}` (3 знака). В `shrinks` — `True`, если значения убывают с ростом `n`.

*Подсказки (2): `dsc.hint('lucky')`*

In [ ]:
%%task ch15/03-experiments/lucky
import numpy as np
from sklearn.metrics import roc_auc_score

### Повторить запуск по журналу · посложнее

В заготовке — журнал `journal` (DataFrame) с тремя запусками: у каждого есть `model`, `params` (строка JSON с параметрами), `seed` и `cv_auc_mean`. Напишите `rebuild(row)`, которая по строке журнала создаёт модель: для `"LogReg"` — `LogisticRegression(**params)`, для `"Forest"` — `RandomForestClassifier(**params, random_state=seed)`. Для каждой строки пересчитайте среднее AUC (`make`, `CV` — в заготовке) и сохраните `check_df` — DataFrame со столбцами `run_id`, `logged`, `reproduced` (4 знака) и `ok` (`abs(разница) < 0.0005`). Один из запусков в журнале записан с ошибкой — его `run_id` положите в `broken`.

*Подсказки (3): `dsc.hint('reproduce')`*

In [ ]:
%%task ch15/03-experiments/reproduce
import json
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def rebuild(row):
    params = json.loads(row["params"])
    ...

### Честная финальная оценка · со звёздочкой

Проведите полный цикл выбора, как в работе: на обучающей части кросс-валидацией (`CV`) выберите лучшую из моделей словаря `candidates` (заготовка), затем **один раз** обучите её на всём `X_train` и оцените на `X_test`. Сохраните `cv_table` — Series средних AUC по именам кандидатов, `chosen` — имя лучшего, `cv_best` (4 знака), `test_auc` (4 знака) и `optimism` = `cv_best − test_auc` (4 знака).

*Подсказки (3): `dsc.hint('ch15/03-experiments/honest')` · задача по желанию*

In [ ]:
%%task ch15/03-experiments/honest
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

## Урок 5. Упаковка и запуск: окружение, Docker, расписание, журнал

«У меня в ноутбуке работает» — самая частая фраза перед тем, как модель не заработала на сервере. Разбираем, как зафиксировать окружение, как устроен пакетный скрипт, который можно спокойно запускать по расписанию, что такое Docker и cron, и как сервис ведёт журнал, по которому потом разбирают любые проблемы. Журнал сервиса оттока за неделю — наш материал для практики и для проекта главы.

Теория — на сайте курса: `ch15/04-deploy.html`. Задачи «собери код из строк» (1) остались на сайте.

### Версии окружения · разминка

Соберите `req_text` — текст для `requirements.txt`: по строке `пакет==версия` для `numpy`, `pandas`, `scikit-learn` (именно в таком порядке, строки через `"\n"`). В `sk_version` — версия scikit-learn.

*Подсказки (2): `dsc.hint('ch15/04-deploy/versions')`*

In [ ]:
%%task ch15/04-deploy/versions
from importlib.metadata import version

### Строки расписания · разминка

Заполните словарь `crons` строками cron для расписаний: `"weekly"` — по понедельникам в 6:30; `"nightly"` — каждый день в 2:15; `"every_10"` — каждые 10 минут; `"workdays_9"` — по будням (пн–пт) в 9:00; `"monthly"` — 1-го числа в 3:00.

*Подсказки (2): `dsc.hint('cron')`*

In [ ]:
%%task ch15/04-deploy/cron
crons = {
    "weekly": "",
    "nightly": "",
    "every_10": "",
    "workdays_9": "",
    "monthly": "",
}

### Пакетный скрипт · база

Модель `MODEL`, паспорт `CARD` и испорченная выгрузка `export.csv` (300 гостей) — в заготовке. Напишите `run_batch(input_path, out_dir, date)` по образцу урока: отложить строки, где `visits_90d < 0`, или `delivery_share` вне 0–1, или `tenure_months < 1` (столбец `reason` — любой непустой текст); годные посчитать; записать `out_dir/scores_<date>.csv` (столбцы `guest_id`, `proba`, `in_campaign`, `model_version`) и `out_dir/rejected_<date>.csv`; вернуть сводку с ключами `n_input`, `n_scored`, `n_rejected`, `n_in_campaign`. Запустите для даты `"2026-12-07"` с папкой `"out"` — сводка в `summary`.

*Подсказки (3): `dsc.hint('batch-job')`*

In [ ]:
%%task ch15/04-deploy/batch-job
import os
import pandas as pd

def run_batch(input_path, out_dir, date):
    df = pd.read_csv(input_path)
    ...

summary = run_batch("export.csv", "out", "2026-12-07")
print(summary)

### Повторный запуск · база

Готовая `run_batch` — в заготовке. Запустите её **дважды** за одну дату `"2026-12-14"` (папка `"out"`) и убедитесь, что файл не задвоился: `rows_after_1`, `rows_after_2` — число строк в `out/scores_2026-12-14.csv` после первого и второго запуска. Затем покажите, как ломается неидемпотентная версия: функция `run_batch_append` (в заготовке) дописывает результаты в общий `out/all_scores.csv`. Удалите этот файл, если он есть, запустите её дважды и сохраните `rows_append` — сколько строк в нём после двух запусков, и `dup_guests` — сколько `guest_id` встречается в нём больше одного раза.

*Подсказки (2): `dsc.hint('idempotent')`*

In [ ]:
%%task ch15/04-deploy/idempotent
import os
import pandas as pd

### Журнал в таблицу · база

Прочитайте `service_log.jsonl` в DataFrame `log` (столбец `ts` — дата-время). Отдельно разложите входы в `payload_df` (`pd.json_normalize`). Сохраните `n_requests`, `status_share` — словарь `{статус: доля}` (3 знака), `clients` — отсортированный список клиентов и `period` — кортеж `(первая дата, последняя дата)` строками `"ГГГГ-ММ-ДД"`.

*Подсказки (2): `dsc.hint('parse-log')`*

In [ ]:
%%task ch15/04-deploy/parse-log
import json
import pandas as pd

### Время ответа и нагрузка · база

Таблица `log` — в заготовке. Для успешных запросов (`status == 200`) посчитайте `p50` и `p95` времени ответа (мс, 1 знак). Посчитайте `by_hour` — число запросов по часу суток (Series с индексом 7…22), `peak_hour` — час с наибольшим числом запросов. Нарисуйте столбчатую диаграмму `by_hour`.

*Подсказки (2): `dsc.hint('latency-hours')`*

In [ ]:
%%task ch15/04-deploy/latency-hours
import pandas as pd
import matplotlib.pyplot as plt

### Записать строку журнала · посложнее

Напишите `log_request(path, request_id, client, status, latency_ms, error="")`: дописать в файл `path` одну строку JSON с полями `ts` (текущее время, `datetime.now().isoformat(timespec="seconds")`), `request_id`, `client`, `status`, `latency_ms`, `error`; русские буквы — без экранирования; строка заканчивается `"\n"`. Удалите `my_log.jsonl`, если он есть, запишите три запроса из заготовки и прочитайте файл в DataFrame `mine`.

*Подсказки (2): `dsc.hint('write-log')`*

In [ ]:
%%task ch15/04-deploy/write-log
import os
import json
from datetime import datetime
import pandas as pd

def log_request(path, request_id, client, status, latency_ms, error=""):
    ...

### Отчёт по дням · посложнее

Таблица `log` — в заготовке. Постройте `daily` — DataFrame с индексом-датой (`log["ts"].dt.date`) и столбцами: `n` — число запросов, `error_rate` — доля ответов не 200 (3 знака), `p95_ms` — 95-й процентиль времени ответа по **успешным** запросам (1 знак). Сохраните `alert_days` — список дат (строки `"ГГГГ-ММ-ДД"`), где `error_rate > 0.02` или `p95_ms > 100`.

*Подсказки (3): `dsc.hint('ch15/04-deploy/daily')`*

In [ ]:
%%task ch15/04-deploy/daily
import pandas as pd

log["date"] = log["ts"].dt.date

### Проверить расписание · со звёздочкой

Напишите `cron_match(expr, dt)`: вернуть `True`, если момент `dt` (`datetime`) подходит под строку cron из пяти полей. Поддержите в каждом поле `*`, число, список через запятую (`1,15`), диапазон (`1-5`) и шаг (`*/15`). День недели — `dt.isoweekday() % 7` (0 — воскресенье, 1 — понедельник). Посчитайте `n_runs` — сколько раз за неделю 2–8 ноября 2026 (каждую минуту с 00:00 2.11 по 23:59 8.11) сработало бы расписание `"*/30 8-10 * * 1-5"`.

*Подсказки (3): `dsc.hint('cron-match')` · задача по желанию*

In [ ]:
%%task ch15/04-deploy/cron-match
from datetime import datetime, timedelta

def field_match(field, value):
    ...

def cron_match(expr, dt):
    ...

## Урок 6. Проект: разбор инцидента в сервисе модели

Проект главы. Понедельник, 9 ноября. За прошлую неделю у сервиса оттока накопились жалобы: приложение «стало тормозить», сайт «перестал показывать предложения», а команда хочет откатить модель к старой версии. По журналу сервиса восстанавливаем, что произошло на самом деле, находим три разных проблемы, проверяем, к чему привёл бы откат, выбираем рекомендацию и сверяем свой вывод с разбором инцидента и планом исправлений.

Теория — на сайте курса: `ch15/05-project.html`.

### Шаг 1. Картина по дням · база

Прочитайте журнал в `log` (`ts` — дата-время). Постройте `daily` — DataFrame по датам со столбцами `n`, `share_500`, `share_422` (доли от всех запросов дня, 3 знака) и `p95_ms` (по успешным запросам, 1 знак). Сохраните `first_slow_day` — первую дату, когда `p95_ms > 100`, и `first_err_day` — первую дату, когда `share_500 > 0.02` (обе — строки `"ГГГГ-ММ-ДД"`).

*Подсказки (3): `dsc.hint('timeline')`*

In [ ]:
%%task ch15/05-project/timeline
import json
import pandas as pd

### Шаг 2. Тормоза · база

Таблица `log` — в заготовке. Для успешных запросов посчитайте `lat` — DataFrame с индексом-версией модели и столбцами `p50`, `p95` (1 знак) и `n`. Сохраните `sla_ok` — словарь `{версия: p95 <= 100}` и `switch_time` — время первого запроса версии 1.1 (строка `"ГГГГ-ММ-ДД ЧЧ:ММ"`). Нарисуйте время ответа успешных запросов по времени (точками), цветом разделив версии.

*Подсказки (2): `dsc.hint('ch15/05-project/slow')`*

In [ ]:
%%task ch15/05-project/slow
import pandas as pd
import matplotlib.pyplot as plt

ok = log[log["status"] == 200]

### Шаг 3. Ошибки сайта · база

Таблица `log` — в заготовке. Найдите ошибки 500: сохраните `err_by_client` — число ошибок 500 по клиентам (Series), `first_500` — время первой ошибки 500 (строка `"ГГГГ-ММ-ДД ЧЧ:ММ"`), `messages` — список уникальных текстов ошибок 500. Затем выясните, чего не хватает во входе: для запросов сайта посчитайте `has_ds_before` и `has_ds_after` — долю запросов, где в `payload` есть ключ `delivery_share`, до и после `first_500` (3 знака). В `web_fail_after` — долю ответов 500 у сайта после `first_500` (3 знака).

*Подсказки (3): `dsc.hint('web500')`*

In [ ]:
%%task ch15/05-project/web500
import pandas as pd

e500 = log[log["status"] == 500]

### Шаг 4. Что прислало приложение 3.2 · посложнее

Таблица `log` — в заготовке. Разложите `payload` успешных запросов в столбцы (`pd.json_normalize`) и добавьте клиента. Сохраните `check_by_client` — медиану `avg_check` по клиентам (Series), `ratio_32` — во сколько раз медиана у `"app 3.2"` больше, чем у `"app 3.1"` (округлить до целого), и `share_over_3000` — долю запросов с `avg_check > 3000` по клиентам (Series, 3 знака). Сравните со средним чеком в обучающих данных: `train_max` — максимальный `avg_check` в `guests_churn.csv`.

*Подсказки (2): `dsc.hint('kopecks')`*

In [ ]:
%%task ch15/05-project/kopecks
import pandas as pd

ok = log[log["status"] == 200].reset_index(drop=True)

### Шаг 5. А если откатиться? · посложнее

Модели `v10`, `v11` и таблица успешных запросов с входами `P` (столбец `client` и все признаки) — в заготовке. Посчитайте решения (`proba >= 1/3`) обеих версий на **этих же входах как есть** и на исправленных (у `app 3.2` средний чек делится на 100). Сохраните `campaign_share` — DataFrame с индексом-клиентом и столбцами `v10_as_is`, `v10_fixed`, `v11_as_is`, `v11_fixed` (доли гостей в акции, 3 знака). Сохраните `rollback_harm` — сколько запросов приложения 3.2 получили бы предложение при откате на 1.0 «как есть», но не получили бы при исправленных данных.

*Подсказки (3): `dsc.hint('rollback')`*

In [ ]:
%%task ch15/05-project/rollback
import pandas as pd

fixed = P.copy()
fixed.loc[fixed["client"] == "app 3.2", "avg_check"] = fixed["avg_check"] / 100

### Шаг 6. Проверка, которая бы спасла · база

Таблица успешных запросов `P` (входы + `client`) — в заготовке. Напишите `range_problems(row)` — список нарушений диапазонов для одной строки входа: `avg_check` — пропуск или 50–3000, `delivery_share` — 0–1, `visits_90d` ≥ 0 (сообщения — любые строки, начинающиеся с имени поля). Примените ко всем строкам `P` и сохраните `would_reject` — долю запросов, которые получили бы 422, по клиентам (Series, 3 знака), `rejected_32` — сколько запросов приложения 3.2 было бы отклонено и `first_bad` — время первого такого запроса (строка `"ГГГГ-ММ-ДД ЧЧ:ММ"`, время — в столбце `ts`).

*Подсказки (3): `dsc.hint('guard')`*

In [ ]:
%%task ch15/05-project/guard
import pandas as pd

def range_problems(row):
    problems = []
    ...
    return problems

### Шаг 7. Быстрее, но не хуже · со звёздочкой

Можно ли сохранить лес 1.1, но уложиться в требование по скорости? Обучите на `X_train` леса `RandomForestClassifier(n, min_samples_leaf=5, random_state=0)` (в конвейере `make`) с `n` из `[150, 60, 20]`. Для каждого посчитайте AUC на `X_test` (4 знака) и медиану времени `predict_proba` на одной строке по первым 40 строкам `X_test` (мс). Сохраните `speed` — DataFrame с индексом `n` и столбцами `auc`, `ms`; `best_small` — наименьшее `n`, у которого AUC не ниже AUC 150 деревьев более чем на 0.002, и `speedup` — во сколько раз `ms` у `best_small` меньше, чем у 150 деревьев.

*Подсказки (3): `dsc.hint('faster')` · задача по желанию*

In [ ]:
%%task ch15/05-project/faster
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rows = []

### Шаг 8. Рекомендация · база

Все цифры собраны — теперь решение за вами. Руководитель CRM ждёт одну рекомендацию. Какую дать? Запишите в `advice` номер варианта:

1. Откатиться на 1.0: она работала до 5 ноября без проблем.
2. Не откатываться. Добавить в сервис проверку входа с диапазонами, выкатить лес из 60 деревьев после замера скорости, а сайт и приложение 3.2 пусть исправляют запросы у себя.
3. Ничего не менять в сервисе: попросить веб-команду вернуть `delivery_share`, остальное само пройдёт.
4. Откатить и приложение 3.2, и модель — к состоянию до 5 ноября.

*Подсказки (1): `dsc.hint('advice')`*

In [ ]:
%%task ch15/05-project/advice
advice = ...
print(advice)

## Урок 7. Финальный проект курса

Финальная работа курса и последний шаг основного пути (после итогового проекта главы 8) — от сырой базы до упакованной модели и рекомендаций в рублях. Направление «Зерно для офиса» теряет корпоративных клиентов и хочет каждый квартал получать список офисов, которым стоит позвонить. Собираете признаки из пяти таблиц CRM по срезам во времени, сравниваете три класса моделей, переводите прогноз в деньги, выпускаете модель как артефакт со скриптом расчёта и оформляете всё блокнотом и репозиторием для портфолио.

Теория — на сайте курса: `ch15/06-final-project.html`.

### Шаг 1. Аудит базы · база

Откройте `office.db` и посчитайте: `tables` — словарь «таблица → число строк» для всех пяти таблиц, `n_test` — число тестовых клиентов, `n_dup_tickets` — сколько строк `tickets` повторяют уже встреченный `ticket_id`, `n_returns` — число заказов со статусом «возврат», `last_order` — последнюю дату заказа строкой `"ГГГГ-ММ-ДД"`.

*Подсказки (3): `dsc.hint('audit')`*

In [ ]:
%%task ch15/06-final-project/audit
import sqlite3
import pandas as pd

con = sqlite3.connect("office.db")
names = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con)["name"].tolist()
print(names)

### Шаг 2. Цель на срезах · база

Таблица `base` — настоящие клиенты с договорами (тестовые уже убраны). Напишите `build_target(cutoff, months=3)`: для клиентов, действующих на дату среза (договор начался не позже среза и не расторгнут на эту дату), вернуть Series «`client_id` → 1, если `end_date` не позже чем через `months` календарных месяцев после среза, иначе 0». Сохраните `rates` — словарь «срез (строка) → (действующих, доля ушедших с 3 знаками)» для срезов `cuts`.

*Подсказки (3): `dsc.hint('target')`*

In [ ]:
%%task ch15/06-final-project/target
# уже есть: base — клиенты с датами договора
import pandas as pd
cuts = ["2025-09-30", "2025-12-31", "2026-03-31", "2026-06-30"]

def build_target(cutoff, months=3):
    c = pd.Timestamp(cutoff)
    ...

### Шаг 3. Признаки из заказов · база

Таблица `orders` и функция `active_at(cutoff)` уже загружены. В `orders` остались заказы тестовых клиентов, но `active_at` возвращает только настоящих, поэтому, если строить признаки по её индексу, тестовые в них не попадут. Напишите `order_features(cutoff)`: по строке на каждого действующего на срез клиента (индекс — `client_id`) и столбцы

- `orders_90` — число заказов за 90 дней до среза включительно (`order_date > срез − 90 дней` и `<= срез`), без отменённых;
- `revenue_90` — их сумма `amount` (возвраты с минусом входят);
- `days_since_order` — дней от последнего неотменённого заказа до среза; если заказов не было — дней с начала договора.

Сохраните `feat = order_features("2026-06-30")` и `means` — словарь средних трёх столбцов (3 знака).

*Подсказки (3): `dsc.hint('ch15/06-final-project/features')`*

In [ ]:
%%task ch15/06-final-project/features
# уже есть: orders, base, active_at(cutoff)
import pandas as pd

def order_features(cutoff):
    c = pd.Timestamp(cutoff)
    act = active_at(c).set_index("client_id")
    f = pd.DataFrame(index=act.index)
    ...
    return f

### Шаг 3б. Утечка через дату · посложнее

Покажите, как выглядит утечка. На срезе 30.06.2026 таблица `test` (признаки и `churn`) готова. Посчитайте ROC-AUC одного признака `days_since_order` (честного, из `test`) — `auc_honest`. Затем постройте «неосторожный» вариант: последний заказ берётся по **всей** таблице `orders` без фильтра по дате (отменённые всё так же не считаем), `days = (срез − последний заказ).days`, клиентам без заказов — 0. Его ROC-AUC — `auc_leaky` (оба — 3 знака). В `n_future` сохраните, сколько неотменённых заказов в таблице датированы позже среза.

*Подсказки (3): `dsc.hint('ch15/06-final-project/leak')`*

In [ ]:
%%task ch15/06-final-project/leak
# уже есть: orders, test (срез 30.06.2026: признаки + churn), функции build_features и др.
import pandas as pd
from sklearn.metrics import roc_auc_score
c = pd.Timestamp("2026-06-30")

### Шаг 5. Три класса моделей · посложнее

Таблицы `train` (три среза) и `test` (30.06.2026), списки признаков `CAT` и `NUM` и функция `_pre()` (one-hot для категорий + масштаб для чисел) готовы. Сравните кросс-валидацией `GroupKFold(n_splits=5)` с группами `train["client_id"]` по ROC-AUC три модели:

- `"logreg"` — `make_pipeline(_pre(), LogisticRegression(max_iter=2000))`;
- `"boosting"` — `make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT)], remainder="passthrough"), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=42))`;
- `"knn"` — `make_pipeline(_pre(), KNeighborsClassifier(n_neighbors=25))`.

Сохраните `cv_auc` — словарь средних (3 знака), `best` — лучшую по кросс-валидации и `test_auc` — ROC-AUC лучшей модели, обученной на всём `train`, на `test` (3 знака).

*Подсказки (3): `dsc.hint('ch15/06-final-project/models')`*

In [ ]:
%%task ch15/06-final-project/models
# уже есть: train, test, CAT, NUM, _pre(), make_pipeline, ColumnTransformer, OneHotEncoder, LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import roc_auc_score

X, y, groups = train[CAT + NUM], train["churn"], train["client_id"]

### Шаг 6. Список звонков в рублях · посложнее

На тесте посчитаны вероятности ухода `test["p"]` (логистическая регрессия, обученная на `train`). Для каждого клиента: месячная выручка `R = revenue_90 / 3`, ожидаемая прибыль звонка `exp = p · 0.5 · 0.35 · 12 · R − 1500 − 0.2 · R`. Список звонков — клиенты с `exp > 0` по убыванию `exp`, не больше 150. Посчитайте **фактическую** прибыль списка (в формуле вместо `p` — настоящий `churn`): `profit_model` (целые рубли), `n_calls` и `caught` — сколько из списка действительно ушли бы. Для сравнения — `profit_rule` (150 клиентов с наименьшим `revenue_trend`) и `profit_all` (позвонить всем).

*Подсказки (3): `dsc.hint('ch15/06-final-project/profit')`*

In [ ]:
%%task ch15/06-final-project/profit
# уже есть: test с колонками p (вероятность ухода), churn, revenue_90, revenue_trend
SAVE, MARGIN, MONTHS, CALL, DISCOUNT = 0.5, 0.35, 12, 1500, 0.2

R = test["revenue_90"] / 3

### Шаг 8. Артефакт и список на IV квартал · со звёздочкой

Соберите то, что получит аналитик B2B. Обучите логистическую регрессию (`make_pipeline(_pre(), LogisticRegression(max_iter=2000))`) на **всех** размеченных срезах — `train` и `test` вместе. Сохраните её в `model.joblib`, загрузите обратно и посчитайте на срезе `snapshot("2026-09-30")` вероятность `p_churn` и ожидаемую прибыль `exp_profit` (формула шага 6). Запишите `call_list.csv` со столбцами `client_id`, `city`, `p_churn`, `exp_profit` — клиенты с `exp_profit > 0` по убыванию, не больше 150, числа округлены (вероятность — 3 знака, прибыль — до рубля). Паспорт — словарь `passport` с ключами `model`, `version` (`"1.0"`), `cutoff` (`"2026-09-30"`), `features` (список `CAT + NUM`), `train_rows` (сколько строк в обучении), `n_calls`, `expected_profit` (сумма `exp_profit` списка, целые рубли), `data_md5` (md5 файла `office.db`) — запишите его в `passport.json`.

*Подсказки (3): `dsc.hint('package')` · задача по желанию*

In [ ]:
%%task ch15/06-final-project/package
# уже есть: train, test, snapshot(cutoff), CAT, NUM, _pre(), make_pipeline, LogisticRegression
import hashlib
import json
import joblib
import pandas as pd

## Готово

Запустите `dsc.progress()` — увидите, какие задачи этого сеанса ещё не решены. Перед тем как закрыть блокнот, полезно сделать **Kernel → Restart Kernel and Run All Cells**: если всё проходит сверху вниз, решения не зависят от случайного порядка запуска ячеек.

In [ ]:
dsc.progress()